# OpenOctopus Equity Analysis — Fabric Notebook

Runs the OpenOctopus agent for a given ticker and saves the markdown report to OneLake.

**Setup checklist (one-time):**
1. Attach Fabric Environment `openoctopus-env` (built from `deploy/fabric/environment.yml`)
2. Attach Lakehouse `openoctopus_lakehouse`
3. Link workspace to Azure Key Vault `kv-openoctopus`
4. Upload repo to Lakehouse at `Files/openoctopus/`
5. Update `KV_URL` and `ABFSS_ROOT` in Cell 2 with your actual values

In [ ]:
# ── Parameters (injected by Fabric Data Pipeline) ─────────────────────────
# Tag this cell as 'parameters' in the notebook UI for pipeline injection
TICKER = "AAPL"                         # Stock ticker symbol
SAVE_TO_LAKEHOUSE = True                # Write report to OneLake
LAKEHOUSE_PATH = "Files/equity_reports" # Path inside attached Lakehouse

In [ ]:
# ── Cell 2: Secret injection from Azure Key Vault ─────────────────────────
# Replaces .env — secrets are read via mssparkutils without storing in notebook
from notebookutils import mssparkutils
import os

# TODO: replace with your Key Vault URL
KV_URL = "https://kv-openoctopus.vault.azure.net/"

AZURE_OAI_ENDPOINT = mssparkutils.credentials.getSecret(KV_URL, "AZURE-OAI-ENDPOINT")
AZURE_OAI_KEY      = mssparkutils.credentials.getSecret(KV_URL, "AZURE-OAI-KEY")
FMP_KEY            = mssparkutils.credentials.getSecret(KV_URL, "FMP-API-KEY")

# Inject into environment so config/settings.py (or fabric shim) picks them up
# BASE_URL points directly at the Azure OpenAI deployment (no api_version in path)
os.environ["OPENAI_API_KEY"] = AZURE_OAI_KEY
os.environ["BASE_URL"]       = f"{AZURE_OAI_ENDPOINT.rstrip('/')}/openai/deployments/gpt-4o"
os.environ["FMP_API_KEY"]    = FMP_KEY
os.environ["MODEL"]          = "gpt-4o"
os.environ["EDGAR_IDENTITY"] = "openoctopus research@yourcompany.com"  # TODO: update

print("Secrets loaded from Key Vault.")

In [ ]:
# ── Cell 3: Mount repo from Lakehouse and import agent code ───────────────
import sys

# TODO: replace with your OneLake ABFSS path
# Format: abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id>/
# Or use the friendly name if your workspace supports it:
ABFSS_ROOT = "abfss://openoctopus_lakehouse@onelake.dfs.fabric.microsoft.com"
REPO_PATH  = f"{ABFSS_ROOT}/Files/openoctopus"

mssparkutils.fs.mount(REPO_PATH, "/openoctopus")
sys.path.insert(0, "/openoctopus")

# Apply Fabric settings shim — neutralises load_dotenv() in config/settings.py
import deploy.fabric.config.fabric_settings as _fs
import config.settings as _s
for _attr in dir(_fs):
    if not _attr.startswith("_"):
        setattr(_s, _attr, getattr(_fs, _attr))

from agent.loop import run_analysis
print("Agent code imported successfully.")

In [ ]:
# ── Cell 4: Run the analysis ───────────────────────────────────────────────
# run_analysis() uses ThreadPoolExecutor internally — runs on Spark driver node
# (pure Python; Spark workers are not used)
print(f"Starting analysis for: {TICKER}")
report_markdown = run_analysis(TICKER)

# Render in notebook output
from IPython.display import Markdown, display
display(Markdown(report_markdown))

In [ ]:
# ── Cell 5: Save report to OneLake ────────────────────────────────────────
import datetime

if SAVE_TO_LAKEHOUSE:
    today    = datetime.date.today().isoformat()
    filename = f"{TICKER}_{today}.md"
    dest     = f"{ABFSS_ROOT}/{LAKEHOUSE_PATH}/{filename}"

    mssparkutils.fs.put(dest, report_markdown, overwrite=True)
    print(f"Report saved to OneLake: {LAKEHOUSE_PATH}/{filename}")
else:
    print("SAVE_TO_LAKEHOUSE=False — report not persisted.")